# Install library

In [ ]:
!pip install -q doclayout-yolo==0.0.3
!pip install -q ultralytics pymupdf opencv-python pillow
!pip install -q numpy
!pip install -q transformers sentencepiece torch

# pytesseract
!pip install -q pytesseract
!apt-get install -y tesseract-ocr tesseract-ocr-vie

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 84.5 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-vie
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 417 kB of archives.
After this operation, 546 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-vie all 1:4.00~git30-7274cfa-1.1 [417 kB]
Fetched 417 kB in 0s (4,004 kB/s)
Selecting previously unselected package tesseract-ocr-vie.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-vie_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking 

# Library

In [ ]:
from huggingface_hub import snapshot_download
import os
import fitz
import cv2
import torch
import numpy as np
from PIL import Image
from doclayout_yolo import YOLOv10
import re
import requests
from urllib.parse import urlparse
import uuid
import json
import pytesseract
from PIL import Image
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM
from openai import OpenAI
import glob

# Load model

In [ ]:
# == download weights ==
model_dir = snapshot_download('juliozhao/DocLayout-YOLO-DocStructBench-imgsz1280-2501', local_dir='./models/DocLayout-YOLO-DocStructBench-imgsz1280-2501')
# == select device ==
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

doclayout_yolo_docstructbench_imgsz1280_(…):   0%|          | 0.00/39.8M [00:00<?, ?B/s]

In [ ]:
MODEL_PATH = "./models/DocLayout-YOLO-DocStructBench-imgsz1280-2501/doclayout_yolo_docstructbench_imgsz1280_2501.pt"

model = YOLOv10(MODEL_PATH)
model.to(DEVICE)

YOLOv10(
  (model): YOLOv10DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, trac

In [ ]:
def download_pdf_from_url(url, save_dir="input", chunk_size=8192):
    os.makedirs(save_dir, exist_ok=True)
    filename = os.path.basename(urlparse(url).path)
    if not filename.endswith(".pdf"):
        filename = "document.pdf"

    save_path = os.path.join(save_dir, filename)
    if os.path.exists(save_path):
        print(f"PDF existed: {save_path}")
        return save_path

    print(f"Downloading PDF to {save_path} ...")

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(chunk_size):
                if chunk:
                    f.write(chunk)

    return save_path

def pdf_to_images(pdf_path, out_dir="pdf_pages", dpi=200):
    os.makedirs(out_dir, exist_ok=True)
    doc = fitz.open(pdf_path)

    image_paths = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=dpi)
        img_path = f"{out_dir}/page_{i+1:03d}.png"
        pix.save(img_path)
        image_paths.append(img_path)

    return image_paths


# OCR Processing

In [ ]:
try:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    OPENAI_AVAILABLE = True
except Exception as e:
    print(f"Warning: OpenAI API not available - {e}")
    OPENAI_AVAILABLE = False

SYSTEM_PROMPT = """
Bạn là hệ thống sửa lỗi OCR cho đề toán tiếng Việt.

NHIỆM VỤ:
- Sửa lỗi chính tả tiếng Việt do OCR
- Sửa ký hiệu toán học:
  + "Z" giữa hai đoạn thẳng → "//" (song song)
  + "L" → "⊥" (vuông góc)
  + "Tính A và C" → "Tính góc A và góc C" (nếu là góc trong hình học)
  + "A = 90°" → "góc A = 90°"
  + "AM ——=——_" → "AM/MD" (phân số)
  + "BN > NC'" → "BN/NC"
  + "——", "___", "===", "|||" → "/" (dấu phân số)
  + Nếu thấy pattern "A ký_tự_lạ B", kiểm tra xem có phải "A/B" không

- KHÔNG thêm hoặc bớt dữ kiện
- KHÔNG giải bài toán
- KHÔNG thêm lời giải thích
- KHÔNG trả về text như "Câu hỏi của bạn không chứa lỗi..."
- CHỈ trả về text đã sửa lỗi
- Giữ nguyên nhãn hình: H.1, (H.2.3), H.5.18

VÍ DỤ:
Input: "AM Z NC" → Output: "AM // NC"
Input: "AM ——=——_ BN" → Output: "AM/MD = BN/NC"
Input: "AM Z NC" → Output: "AM // NC"
Input: "Tính A và C" → Output: "Tính góc A và góc C"
Input: "AB L CD" → Output: "AB ⊥ CD"
"""

In [ ]:
import unicodedata

def normalize_math_symbols(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)

    text = re.sub(r'//+', ' song song ', text)

    text = text.replace('⊥', ' vuông góc ')

    text = text.replace('∠', ' góc ')

    text = re.sub(r'(\d+)\s*°', r'\1 độ', text)

    text = text.replace('≡', ' bằng ')
    text = text.replace('≅', ' đồng dư ')
    text = text.replace('∼', ' đồng dạng ')
    text = text.replace('~', ' đồng dạng ')
    text = text.replace('≠', ' khác ')
    text = text.replace('≥', ' lớn hơn hoặc bằng ')
    text = text.replace('≤', ' nhỏ hơn hoặc bằng ')
    text = text.replace('√', ' căn bậc hai của ')
    text = text.replace('π', ' pi ')
    text = text.replace('∞', ' vô cùng ')

    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
def fix_ocr_text(text: str) -> str:
    if not text.strip():
        return ""

    if OPENAI_AVAILABLE:
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                temperature=0,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text}
                ]
            )
            result = response.choices[0].message.content.strip()

            # Filter out unwanted responses
            if "câu hỏi của bạn" in result.lower() or "không chứa lỗi" in result.lower():
                return text.strip()
            result = normalize_math_symbols(result)
            return result
        except Exception as e:
            print(f"OpenAI API error: {e}, using basic cleaning")

    # Fallback: basic text cleaning
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
def ocr_block(img_crop):
    gray = cv2.cvtColor(img_crop, cv2.COLOR_BGR2GRAY)

    # tăng tương phản nhẹ
    gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)

    config = r"--oem 3 --psm 6 -l vie+eng"
    text = pytesseract.image_to_string(gray, config=config)

    return text.strip()

In [ ]:
# pytesseract
def detect_and_crop_text(
    image_path,
    conf=0.25
):

    img = cv2.imread(image_path)

    results = model.predict(
        source=image_path,
        imgsz=1280,
        conf=conf,
        device=DEVICE,
        verbose=False
    )[0]

    names = model.names
    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()
    scores = results.boxes.conf.cpu().numpy()

    text_blocks = []

    for box, cls, score in zip(boxes, classes, scores):
        if names[int(cls)] != "plain text":
            continue

        x1, y1, x2, y2 = map(int, box)
        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        text = ocr_block(crop)
        text = fix_ocr_text(text)

        text_blocks.append({
            "text": text,
            "bbox": [x1, y1, x2, y2],
            "y_top": y1
        })

    text_blocks.sort(key=lambda x: x["y_top"])
    return text_blocks

In [ ]:
ANCHOR_REGEX = re.compile(
    r"""
    ^\s*(
        (\d+\.\d+) |            # 3.28
        (bài\s*\d+) |           # Bài 3
        (câu\s*\d+) |           # Câu 2
        (ví\s*dụ\s*\d*) |       # Ví dụ
        (h\.\d+\.\d+)           # H.2.1
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

In [ ]:
def group_text_blocks_into_problems(text_blocks):
    problems = []
    current_problem = None

    text_blocks = sorted(text_blocks, key=lambda x: x["y_top"])

    for block in text_blocks:
        text = block["text"].strip()
        if not text:
            continue

        match = ANCHOR_REGEX.search(text)

        if match:
            # Đây là đề mới
            if current_problem:
                problems.append(current_problem)

            title = match.group(0).strip()

            current_problem = {
                "id": str(uuid.uuid4()),
                "title": title,
                "blocks": [block],
                "found_solution": False  # Flag để dừng thêm blocks
            }
        else:
            if current_problem and not current_problem.get("found_solution", False):
                # Kiểm tra xem có phải lời giải không
                title_num = current_problem['title'].strip('.')

                # Pattern lời giải cải thiện
                solution_keywords = [
                    r'^\s*\(H\.\d+\.\d+\)',  # (H.3.6) - Reference hình trong lời giải
                    r'^\s*(ta có|xét|do đó|vậy|suy ra|từ đó|chứng minh|giải|nếu)',
                    r'^\s*[a-d]\)',  # a), b), c), d)
                ]

                is_solution = any(re.match(pattern, text, re.IGNORECASE) for pattern in solution_keywords)

                if is_solution:
                    current_problem["found_solution"] = True
                else:
                    current_problem["blocks"].append(block)

    if current_problem:
        problems.append(current_problem)

    for p in problems:
        # Clean up flag
        p.pop("found_solution", None)

        p["content"] = "\n".join(
            b["text"] for b in sorted(p["blocks"], key=lambda x: x["y_top"])
        )

    return problems

In [60]:
def extract_figure_references(text):
    """
    Extract figure references with multiple patterns to handle OCR errors:
    - H.3.2, (H.3.2) - standard
    - H.3.10a, (H.3.10b) - with letter suffix
    - hình 5.5, hình 5.6 - lowercase without H.
    - h.3.1, (h.3.10a) - lowercase h
    """
    figures = []

    # Pattern 1: Standard H.x.x with optional letter suffix
    # Matches: H.3.2, (H.3.10a), H.5.7b
    pattern1 = r'\(?H\.\d+(?:\.\d+)?[a-z]?\)?'
    figures.extend(re.findall(pattern1, text, re.IGNORECASE))

    # Pattern 2: "hình x.x" format
    pattern2 = r'hình\s+(\d+\.\d+)'
    hinh_matches = re.findall(pattern2, text, re.IGNORECASE)
    # Convert to H.x.x format
    figures.extend([f"H.{m}" for m in hinh_matches])

    # Normalize: Remove duplicates and parentheses
    normalized = []
    for fig in figures:
        clean = fig.strip('()').strip()
        # Normalize to uppercase H
        if clean.lower().startswith('h.'):
            clean = 'H.' + clean[2:]
        normalized.append(clean)

    return list(set(normalized))

In [61]:
def deduplicate_problems(problems):
    """
    Logic:
    1. Lần 1: Đề bài (page thấp)
    2. Lần 2: Solution (page cao)
    3. Extract H.x.x từ CẢ 2
    4. Merge figures từ đề + solution
    """
    seen = {}

    for prob in problems:
        title = prob['title']
        page = prob['page']
        content = prob.get('content', '')

        if title not in seen:
            # Lần đầu = ĐỀ BÀI
            figures = extract_figure_references(content)

            seen[title] = {
                'problem': prob,
                'problem_figures': figures,
                'solution': None,
                'solution_page': None,
                'solution_figures': []
            }
        else:
            # Lần 2 = SOLUTION
            existing = seen[title]
            solution_figures = extract_figure_references(content)

            existing['solution'] = content
            existing['solution_page'] = page
            existing['solution_figures'] = solution_figures

            # if solution_figures:
            #     print(f"[{title}] Found {len(solution_figures)} figures in SOLUTION (page {page}): {solution_figures}")

    # Build final result
    result = []
    for title, data in seen.items():
        prob = data['problem']

        # Merge figures từ đề + solution
        all_figures = list(set(data['problem_figures'] + data['solution_figures']))

        # Build final problem
        final_problem = {
            'id': prob['id'],
            'title': prob['title'],
            'page': prob['page'],
            'content': prob['content'],  # ĐỀ BÀI
            'solution': data['solution'],  # LỜI GIẢI
            'solution_page': data['solution_page'],
            'figures': all_figures,  # Merge từ cả 2
            'figures_from_problem': data['problem_figures'],
            'figures_from_solution': data['solution_figures'],
            'blocks': prob.get('blocks', [])
        }

        result.append(final_problem)

    return result

In [62]:
def extract_problems_from_pdf(pdf_path, output_root="output"):
    pages_dir = os.path.join(output_root, "pages")
    problems_dir = os.path.join(output_root, "problems")
    os.makedirs(pages_dir, exist_ok=True)
    os.makedirs(problems_dir, exist_ok=True)

    page_images = pdf_to_images(pdf_path, pages_dir)

    all_problems = []

    for page_idx, page_img in enumerate(page_images):
        text_blocks = detect_and_crop_text(page_img)

        problems = group_text_blocks_into_problems(text_blocks)

        for p in problems:
            problem = {
                "id": str(uuid.uuid4()),
                "title": p["title"],
                "page": page_idx + 1,
                "content": "\n".join(
                    [b["text"] for b in p["blocks"] if b.get("text")]
                ),
            }

            all_problems.append(problem)

    all_problems = deduplicate_problems(all_problems)

    output_json_path = os.path.join(problems_dir, "problems.json")
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(all_problems, f, ensure_ascii=False, indent=2)


    total = len(all_problems)
    with_solution = sum(1 for p in all_problems if p.get('solution'))
    with_figures = sum(1 for p in all_problems if p.get('figures'))
    from_solution = sum(1 for p in all_problems if p.get('figures_from_solution'))
    from_problem = sum(1 for p in all_problems if p.get('figures_from_problem'))

    print(f"   Total problems: {total}")
    print(f"   With solution: {with_solution} ({with_solution/total*100:.1f}%)")
    print(f"   With figures: {with_figures} ({with_figures/total*100:.1f}%)")
    print(f"   └─ From solution: {from_solution}")
    print(f"   └─ From problem: {from_problem}")
    print(f"   Saved to: {output_json_path}")

    return all_problems

# Extract PDF

In [ ]:
# PDF_URL = "https://84864e12bc.vws.vegacdn.vn//data/doc/2025/thcslienninh/2025_2/26/sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf"
# save_dir = "./input"

# pdf_file_path = download_pdf_from_url(PDF_URL, save_dir=save_dir)
# problems = extract_problems_from_pdf(pdf_file_path)

# for d in problems[:5]:
#     print(d)


In [63]:
PDF_URL = "https://84864e12bc.vws.vegacdn.vn//data/doc/2025/thcslienninh/2025_2/26/sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf"
save_dir = "./input"

pdf_file_path = download_pdf_from_url(PDF_URL, save_dir=save_dir)
problems = extract_problems_from_pdf(pdf_file_path)

PDF existed: ./input/sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf
   Total problems: 136
   With solution: 125 (91.9%)
   With figures: 43 (31.6%)
   └─ From solution: 26
   └─ From problem: 17
   Saved to: output/problems/problems.json


In [ ]:
for d in problems[:10]:
    print(d)

In [ ]:
# import shutil

# # Nén folder pages thành zip
# shutil.make_archive('output_pages', 'zip', 'output/pages')
# print("Đã nén xong: output_pages.zip")

# # Download file zip
# from google.colab import files
# files.download('output_pages.zip')

Đã nén xong: output_pages.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# # Save file sau khi extract (Đỡ mất thời gian extract lại vì quá lâu)
# PROBLEMS_FILE = "output/problems/problems.json"

# with open(PROBLEMS_FILE, "r", encoding="utf-8") as f:
#     problems = json.load(f)

# print(f"Loaded {len(problems)} problems from {PROBLEMS_FILE}")

Loaded 250 problems from output/problems/problems.json


# Filtering geometry question

In [65]:
FIGURE_PATTERN = re.compile(
    r'\(?H\.\d+(?:\.\d+)?\)?|'  # H.2.3, (H.5.4)
    r'hình\s+\d+\.\d+|'          # hình 2.3
    r'hình\s+vẽ|'                # hình vẽ
    r'trong\s+hình|'             # trong hình
    r'theo\s+hình|'              # theo hình
    r'như\s+hình',               # như hình
    re.IGNORECASE
)

GEOMETRY_KEYWORDS = [
    "hình", "góc", "đường", "điểm", "đoạn thẳng",
    "tam giác", "tứ giác", "hình vuông", "hình chữ nhật",
    "hình bình hành", "hình thoi", "hình thang",
    "đường tròn", "chu vi", "diện tích", "thể tích",
    "trung điểm", "song song", "vuông góc", "đối xứng",
    "cạnh", "đỉnh", "bán kính", "đường kính",
    "độ dài", "chiều cao", "chiều rộng",
    "lăng trụ", "hình chóp", "tứ diện"
]

In [66]:
def has_figure_reference(problem):
    """Kiểm tra xem bài toán có tham chiếu đến hình vẽ không"""
    # Kiểm tra trong title
    if FIGURE_PATTERN.search(problem.get("title", "")):
        return True
    # Kiểm tra trong content
    if FIGURE_PATTERN.search(problem.get("content", "")):
        return True
    return False

def is_geometry_problem(problem):
    """Kiểm tra xem có phải bài toán hình học không"""
    content = problem.get("content", "").lower()
    # Kiểm tra từ khóa hình học
    return any(keyword in content for keyword in GEOMETRY_KEYWORDS)

def filter_geometry_with_figures(problems):
    """
    Lọc TẤT CẢ bài hình học (không phụ thuộc H.x.x trong đề)
    Matching sẽ quyết định có diagram hay không
    """
    filtered = []

    for problem in problems:
        is_geo = is_geometry_problem(problem)
        has_fig_ref = has_figure_reference(problem)

        # Lấy TẤT CẢ bài hình học
        if is_geo:
            problem['has_figure_hint'] = has_fig_ref  # Flag metadata
            filtered.append(problem)

    return filtered

In [67]:
geometry_problems = filter_geometry_with_figures(problems)

print(f"Total questions: {len(problems)}")
print(f"Number of geometry problems with figures: {len(geometry_problems)}\n")

for i, prob in enumerate(geometry_problems[:5]):
    print(f"[{i+1}] {prob['title']} - Page {prob['page']}")

    # In content
    print(f"Content:")
    print(prob['content'][:500])

    # Tìm và in các figure reference
    content = prob.get('content', '')
    figures = re.findall(r'\(?H\.\d+(?:\.\d+)?\)?', content, re.IGNORECASE)
    if figures:
        print(f"Figures: {', '.join(set(figures))}")

    # In đường dẫn ảnh nếu có
    page_img = f"output/pages/page_{prob['page']:03d}.png"
    if os.path.exists(page_img):
        print(f"Image: {page_img}")
        print("")

Total questions: 136
Number of geometry problems with figures: 64

[1] 1.24 - Page 16
Content:
1.24. a) Tìm đơn thức M biết rằng 2,7x2/4z? : M = 0,9x2yz; b) Biết (-2 xyz) * N = x'# song song z?. Hãy tìm đơn thức N.
Image: output/pages/page_016.png

[2] 1.27 - Page 18
Content:
1.27. Một hình lăng trụ đứng có đáy là một tam giác với ba cạnh bằng 3x, 4x và 5x
(biết rằng đó là một tam giác vuông), chiều cao của hình lăng trụ bằng y
(x> 0, y> 0). Hãy tìm đa thức với hai biến x và y biểu thị diện tích toàn phần
(tổng diện tích xung quanh và diện tích hai đáy) của hình lăng trụ đó. Xác định
bậc của đa thức tìm được.
Image: output/pages/page_018.png

[3] 2.12 - Page 24
Content:
2.12. Từ một khối lập phương có độ dài cạnh là x + 3 (cm), ta cắt bỏ một khối lập phương có độ dài x - 1 (cm) (H.2.3). Tính thể tích phần còn lại, viết kết quả dưới dạng đa thức.
Figures: (H.2.3)
Image: output/pages/page_024.png

[4] 2.24 - Page 30
Content:
2.24. Từ một miếng bia có dạng hình tròn (H.2.4) với bán kính R

In [68]:
def clean_invalid_figure_refs(content, valid_figures):
    """
    Remove figure references không có trong list valid_figures

    Args:
        content: Text content của problem
        valid_figures: List các figure refs hợp lệ (VD: ["H.2.3", "H.3.6"])

    Returns:
        Cleaned content
    """
    all_refs = re.findall(r'\(H\.\d+\.\d+\)|H\.\d+\.\d+', content, re.IGNORECASE)

    for ref in all_refs:
        # Clean parentheses để so sánh
        clean_ref = ref.strip('()')

        # Nếu ref không có trong valid_figures → remove
        if not any(clean_ref.lower() in vf.lower() for vf in valid_figures):
            content = content.replace(ref, '')
            content = content.replace(f'. {ref}', '')  # ". (H.3.6)" → ""

    # Clean up multiple spaces và newlines
    content = re.sub(r'\s+', ' ', content)
    content = re.sub(r'\n\s*\n', '\n', content)
    return content.strip()

# Mapping figure images

In [69]:
def find_figure_images(page_num, figure_refs, diagrams_dir="output/diagrams", is_geometry=False):
    """
    Strategy:
    1. Exact H.x.x match (H.3.2 == H.3.2.png)
    2. Fuzzy match (H3.2, h.3.2, (H.3.2))
    3. Page-based: Tìm diagrams trong ±1 page
    4. Unmatched diagram pool: Lấy diagrams chưa được match gần page này
    """
    matched_images = []

    if not os.path.exists(diagrams_dir):
        return matched_images

    all_images = glob.glob(os.path.join(diagrams_dir, "*.png"))

    # STEP 1: Exact & Fuzzy H.x.x match
    if figure_refs:
        for fig_ref in figure_refs:
            # Clean reference: (H.3.17) -> h.3.17
            clean_ref = fig_ref.strip('()').strip().lower()

            for img_path in all_images:
                img_base = os.path.basename(img_path).lower().replace('.png', '')

                # Exact match: h.3.2 == h.3.2
                if clean_ref == img_base:
                    matched_images.append(img_path)
                    break

                # Fuzzy match: Normalize dots/separators and check EXACT equality
                # This handles h.3.2 == h_3_2 but NOT h.3.2 == h.3.25
                clean_img_no_sep = img_base.replace('.', '').replace('_', '').replace('-', '')
                clean_ref_no_sep = clean_ref.replace('.', '').replace('_', '').replace('-', '')

                # EXACT match after removing separators (no substring matching!)
                if clean_ref_no_sep == clean_img_no_sep:
                    matched_images.append(img_path)
                    break

    # STEP 2: Page-based fallback
    if not matched_images and is_geometry:
        # Tìm diagrams có metadata page trong tên file
        # VD: H.3.2_page_25.png, diagram_page_025.png
        for img_path in all_images:
            img_name = os.path.basename(img_path).lower()

            # Check page trong filename
            page_patterns = [
                f"page_{page_num:03d}",  # page_025
                f"page{page_num}",       # page25
                f"p{page_num}",          # p25
            ]

            if any(pattern in img_name for pattern in page_patterns):
                matched_images.append(img_path)

        # Nếu vẫn không có, lấy diagrams trong ±1 page
        if not matched_images:
            for offset in [-1, 0, 1]:
                target_page = page_num + offset
                if target_page < 1:
                    continue

                for img_path in all_images:
                    img_name = os.path.basename(img_path).lower()
                    page_patterns = [
                        f"page_{target_page:03d}",
                        f"page{target_page}",
                    ]

                    if any(pattern in img_name for pattern in page_patterns):
                        matched_images.append(img_path)

                if matched_images:
                    break  # Tìm thấy rồi --> dừng

    return list(set(matched_images))

In [70]:
def add_figure_images_to_problems(problems, diagrams_dir="output/diagrams"):
    """
    Thêm đường dẫn ảnh hình vẽ vào mỗi bài toán
    """
    for prob in problems:
        page_num = prob['page']

        # Extract figure references từ content nếu chưa có
        if 'figures' not in prob:
            prob['figures'] = extract_figure_references(prob.get('content', ''))

        figure_refs = prob.get('figures', [])

        # Tìm ảnh hình vẽ
        figure_images = find_figure_images(page_num, figure_refs, diagrams_dir)

        prob['figure_images'] = figure_images
        prob['has_figure_images'] = len(figure_images) > 0

    return problems

In [71]:
# Áp dụng matching cho geometry_problems
geometry_problems_with_images = add_figure_images_to_problems(
    geometry_problems,
    diagrams_dir="output/diagrams"
)

for i, prob in enumerate(geometry_problems_with_images[:5], 1):
    print(f"[{i}] {prob['title']} (Page {prob['page']})")
    print(f"    Figure refs: {prob.get('figures', [])}")
    print(f"    Matched images: {prob.get('figure_images', [])}")
    print(f"    Has images: {prob.get('has_figure_images', False)}")
    print()

# Statistics
with_images = sum(1 for p in geometry_problems_with_images if p['has_figure_images'])
print(f"\nProblems with figure images: {with_images}/{len(geometry_problems_with_images)}")

[1] 1.24 (Page 16)
    Figure refs: []
    Matched images: []
    Has images: False

[2] 1.27 (Page 18)
    Figure refs: []
    Matched images: []
    Has images: False

[3] 2.12 (Page 24)
    Figure refs: ['H.2.3']
    Matched images: ['output/diagrams/H.2.3.png']
    Has images: True

[4] 2.24 (Page 30)
    Figure refs: ['H.2.4']
    Matched images: ['output/diagrams/H.2.4.png']
    Has images: True

[5] 3.1 (Page 32)
    Figure refs: []
    Matched images: []
    Has images: False


Problems with figure images: 27/64


# Save json have figures

In [72]:
def clean_title_from_content(content, title):
    pattern = rf'^\s*{re.escape(title)}\.\s*'
    cleaned = re.sub(pattern, '', content, count=1)
    return cleaned.strip()

In [73]:
structured_problems = []

for prob in geometry_problems_with_images:
    if not prob.get('has_figure_images', False):
        continue

    # Extract figures từ matched images
    figures = []
    for img_path in prob.get('figure_images', []):
        img_name = os.path.basename(img_path).replace('.png', '')
        figures.append(img_name)

    valid_figures = figures

    cleaned_content = clean_invalid_figure_refs(prob['content'], valid_figures)

    cleaned_content = clean_title_from_content(cleaned_content, prob['title'])

    structured_data = {
        "stt": len(structured_problems) + 1,
        "problem_id": prob['id'],
        "title": prob['title'],
        "page": prob['page'],
        "content": cleaned_content,
        "figures": figures,
        "page_image": f"output/pages/page_{prob['page']:03d}.png",
        "figure_images": prob.get('figure_images', []),
        "has_figure_images": True
    }

    structured_problems.append(structured_data)

OUTPUT_STRUCTURED = "output/problems/geometry_problems_with_images.json"

with open(OUTPUT_STRUCTURED, "w", encoding="utf-8") as f:
    json.dump(structured_problems, f, ensure_ascii=False, indent=2)

print(f"Saved {len(structured_problems)} problems with figure images to: {OUTPUT_STRUCTURED}")

Saved 27 problems with figure images to: output/problems/geometry_problems_with_images.json


In [74]:
print("\nSample cleaned problems:")
for prob in structured_problems[:10]:
    print(f"\n{prob['title']} - Page {prob['page']}")
    print(f"  Figures: {prob['figures']}")
    print(f"  Content preview: {prob['content'][:150]}...")
    print(f"  Images: {[os.path.basename(img) for img in prob['figure_images']]}")


Sample cleaned problems:

2.12 - Page 24
  Figures: ['H.2.3']
  Content preview: Từ một khối lập phương có độ dài cạnh là x + 3 (cm), ta cắt bỏ một khối lập phương có độ dài x - 1 (cm) (H.2.3). Tính thể tích phần còn lại, viết kết ...
  Images: ['H.2.3.png']

2.24 - Page 30
  Figures: ['H.2.4']
  Content preview: Từ một miếng bia có dạng hình tròn (H.2.4) với bán kính R(cm), người ta khoét một hình tròn ở giữa có bán kính r(cm), r < R....
  Images: ['H.2.4.png']

3.5 - Page 32
  Figures: ['H.3.6']
  Content preview: Cho tứ giác ABCD với AB = BC, CD = DA, góc B = 100 độ, góc D = 120 độ. Tính góc A và góc C....
  Images: ['H.3.6.png']

3.9 - Page 34
  Figures: ['H.3.7']
  Content preview: Cho tam giác ABC vuông cân tại đỉnh A. Ghép thêm vào phía ngoài tam giác đó tam giác BCD vuông cân tại đỉnh B. Chứng minh tứ giác ABDC là một hình tha...
  Images: ['H.3.7.png']

3.10 - Page 34
  Figures: ['H.3.8']
  Content preview: Cho hình thang cân ABCD với hai đường thẳng chứa hai cạnh bên AD, BC 

# Evaluation

In [75]:
def evaluate_matching_pipeline(geometry_problems_with_images, diagrams_dir="output/diagrams"):
    # Problem metrics
    total_problems = len(geometry_problems_with_images)
    matched_problems = sum(1 for p in geometry_problems_with_images if p['has_figure_images'])

    # Diagram metrics
    all_diagrams = glob.glob(os.path.join(diagrams_dir, "*.png"))
    total_diagrams = len(all_diagrams)

    matched_diagram_paths = set()
    for prob in geometry_problems_with_images:
        matched_diagram_paths.update(prob.get('figure_images', []))

    matched_diagrams = len(matched_diagram_paths)

    # Expected refs từ filenames
    expected_refs = set()
    for img_path in all_diagrams:
        img_name = os.path.basename(img_path).replace('.png', '')
        match = re.match(r'(H\.\d+\.\d+)', img_name, re.IGNORECASE)
        if match:
            expected_refs.add(match.group(1).upper())

    # Extracted refs từ OCR
    extracted_refs = set()
    for prob in geometry_problems_with_images:
        extracted_refs.update([f.strip('()').upper() for f in prob.get('figures', [])])

    # Calculate metrics
    match_rate = matched_problems / total_problems if total_problems > 0 else 0
    diagram_utilization = matched_diagrams / total_diagrams if total_diagrams > 0 else 0

    tp = len(expected_refs & extracted_refs)
    fn = len(expected_refs - extracted_refs)
    fp = len(extracted_refs - expected_refs)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Print results
    print(f"Match Rate: {match_rate:.2f}", f"({matched_problems}/{total_problems} problems have diagrams)")

    print(f"Diagram Utilization: {diagram_utilization:.2}", f"({matched_diagrams}/{total_diagrams} diagrams are used)")
    print(f"OCR Precision: {precision:.2}", f"({tp} correct / {tp + fp} extracted)")
    print(f"OCR Recall: {recall:.2}", f"({tp} found / {tp + fn} expected)")
    print(f"F1 Score: {f1:.2}")

    return {
        "match_rate": match_rate,
        "diagram_utilization": diagram_utilization,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

metrics = evaluate_matching_pipeline(geometry_problems_with_images)

Match Rate: 0.42 (27/64 problems have diagrams)
Diagram Utilization: 0.6 (27/45 diagrams are used)
OCR Precision: 0.75 (27 correct / 36 extracted)
OCR Recall: 0.64 (27 found / 42 expected)
F1 Score: 0.69
